In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_df.csv")
print(df.shape)
df[["cpe", "total_spend", "total_budget_spent", "total_engagements",
    "actual_spend_per_engagement"]].head(10)

(417, 50)


,cpe,total_spend,total_budget_spent,total_engagements,actual_spend_per_engagement
0,39.0,1.343553e+08,173511.0,4449,30198.99
1,39.0,1.066785e+07,45942.0,1178,9055.90
2,39.0,5.910300e+07,66417.0,1703,34705.23
3,39.0,1.244877e+08,43173.0,1107,112455.05
4,3.0,8.924099e+08,348081.0,116027,7691.40
5,3.0,2.169626e+08,21084.0,7028,30871.17
6,3.0,4.869855e+07,8379.0,2793,17435.93
7,39.0,1.614190e+09,1055964.0,27076,59617.02
8,39.0,2.061418e+07,17238.0,442,46638.41
9,3.0,5.268067e+07,9117.0,3039,17334.87


In [2]:
# data error flag (don't delete without telling your team)
df["cpe_outlier"] = (df["cpe"] < 1).astype(int)

# sub-product features (main drivers of CPE)
tier_map = {"EMAIL ALERT": "email", "BRANDSPOT": "premium",
            "CONDITION MESSAGE": "other"}
df["product_tier"] = df["sub_product_name"].map(tier_map).fillna("standard")
df["is_email_alert"] = (df["sub_product_name"] == "EMAIL ALERT").astype(int)
df["is_brandspot"] = (df["sub_product_name"] == "BRANDSPOT").astype(int)
df["is_target_list"] = (df["sub_product_name"] == "TARGET LIST MEDIA").astype(int)

# other features
df["unknown_targeting"] = (df["media_type"] == "unknown").astype(int)
df["has_channel_tag"] = (df["channel_tag"] != "none").astype(int)
df["is_us"] = (df["country"] == "USA").astype(int)
df["budget_per_day"] = df["total_budget_spent"] / df["campaign_age_days"].clip(lower=1)
df["budget_per_reach"] = df["total_budget_spent"] / df["unique_reach"]

df.isna().sum().sum()

np.int64(0)

In [3]:
print(df.groupby("product_tier")["cpe"].agg(["mean", "nunique", "count"]))

                   mean  nunique  count
product_tier                           
email          3.000000        1    145
other         11.000000        1      1
premium       68.000000        1     27
standard      38.840246        2    244


In [4]:
df.to_csv("featured_df.csv", index=False)